# loss_03 — L2 Risk-Based Pricing Strategy

**Owner:** Yumeng · **Layer:** L2 (Risk-Based Pricing + RAROC)

**Core question:** *What rate should each loan carry to cover funding cost, operating cost, expected loss, and target margin?*

## Data Sources
| File | Source | Purpose |
|---|---|---|
| `l1_el_breakdown.csv` | Shuxin (loss_02) | 2018Q4 active loan EAD / segment counts / L1 EL benchmark by grade × term × purpose |
| `pd_predictions.csv` | Modeling team | Direct loan-level predicted PD and actual interest rate (source of pricing PD) |
| `accepted_2007_to_2018Q4.csv` | Raw data | Optional fallback source for missing interest-rate fields |

## Why 2018Q4 Active Portfolio
We use the 2018Q4 active portfolio as a **risk-return snapshot** to evaluate whether
historical pricing adequately compensated investors for expected loss. The resulting
pricing rules are intended for **future origination / portfolio selection**, not literal
repricing of already issued fixed-rate loans.


## 0. Imports & Style

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import stats
from matplotlib.patches import Patch

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})

NAVY   = "#1B3A6B"
TEAL   = "#2E8B8B"
ORANGE = "#E07B39"
RED    = "#C0392B"
GREEN  = "#27AE60"
GRAY   = "#95A5A6"

print("Libraries loaded.")


## 1. Paths & Constants

### Parameter rationale
| Parameter | Value | Basis |
|---|---|---|
| `FUNDING_COST` | 3.5% | 3-yr Treasury (~2.5%) + LC platform spread (~1%), 2016-2018 |
| `OP_COST` | 1.0% | LC asset-light model; traditional banks ~2-3% |
| `TARGET_MARGIN` | 1.5% | Mid-range of typical consumer lending margin (1-2%) |
| `HURDLE_RATE` | 12% | Project convention (framework one-pager), aligned with L4 |
| `USURY_CAP` | 30% | Conservative proxy across US state usury laws (18-36% range) |
| `REPRICE_THRESH` | -50bp | Gaps within ±50bp treated as noise / model uncertainty |
| `GROW_THRESH` | +50bp | Same — industry practice typically 25-100bp |
| `K_VaR` | 3.5 | Basel IRB approximation for 99.9% VaR confidence |
| `MIN_N` | 30 | Minimum loans per segment for reliable confidence interval |

In [ ]:
L1_BREAKDOWN = Path("data/processed/l1_el_breakdown.csv")
PD_FILE      = Path("data/pd_predictions.csv")
RAW_PATH     = Path("/Users/yumengfang/Desktop/583_project/Data/accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv")
OUT_DIR      = Path("data/processed")
FIG_DIR      = Path("outputs/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

LGD_CONST = 0.9013
FUNDING_COST   = 0.035
OP_COST        = 0.010
TARGET_MARGIN  = 0.015
HURDLE_RATE    = 0.12
USURY_CAP      = 0.30
REPRICE_THRESH = -0.005
GROW_THRESH    =  0.005
K_VaR          = 3.5
MIN_N          = 30
CI_LEVEL       = 0.95
Z_SCORE        = stats.norm.ppf(1 - (1 - CI_LEVEL) / 2)

ACTIVE_STATUSES = ["Current", "In Grace Period", "Late (16-30 days)", "Late (31-120 days)"]
PD_COL_CANDIDATES = [
    "pd_12m", "PD_12m", "predicted_pd", "default_probability",
    "pd", "PD", "pred_pd", "prob_default", "p_default",
    "y_pred_proba", "default_prob", "predicted_probability", "probability_default"
]
EAD_COL_CANDIDATES = ["out_prncp", "funded_amnt", "loan_amnt"]


## 2. Data Loading & Preparation

### Strategy
- Segment frame from `l1_el_breakdown.csv` (`grade`, `term_num`, `purpose`, `ead`, `n`).
- Rename L1 EL to `el_12m_l1`; keep only as reference.
- Segment-level `avg_pd` is aggregated from loan-level predicted PD in `pd_predictions.csv`.
- Segment-level rate is aggregated from loan-level `int_rate` in `pd_predictions.csv`.


In [ ]:
df = pd.read_csv(L1_BREAKDOWN)
if "el_12m" in df.columns:
    df = df.rename(columns={"el_12m": "el_12m_l1"})

print(f"L1 breakdown : {len(df):,} segments")
print(f"Total EAD    : ${df['ead'].sum()/1e9:.2f}B")
if "el_12m_l1" in df.columns:
    print(f"L1 EL ref    : ${df['el_12m_l1'].sum()/1e6:.1f}M")


In [ ]:
def parse_percent_or_decimal(series, col_name="value"):
    s = series.astype(str).str.replace("%", "", regex=False).str.strip()
    x = pd.to_numeric(s, errors="coerce")
    x = pd.Series(x, index=series.index, name=col_name)
    return x.where(x <= 1, x / 100)


def detect_pd_column(pdf):
    cols = set(pdf.columns)
    explicit_priority = ["pd_12m", "PD_12m", "predicted_pd", "default_probability"]
    for c in explicit_priority:
        if c in cols:
            return c
    for c in PD_COL_CANDIDATES:
        if c in cols:
            return c
    raise ValueError(f"No PD column found. Available columns: {list(pdf.columns)}")

pd_df = pd.read_csv(PD_FILE)
print(f"pd_predictions columns: {pd_df.columns.tolist()}")
PD_COL = detect_pd_column(pd_df)
print(f"Detected PD column: {PD_COL}")

pd_df["term_num"] = pd_df["term"].astype(str).str.extract(r"(\d+)").astype(float)
pd_df["int_rate_dec"] = parse_percent_or_decimal(pd_df["int_rate"], "int_rate_dec")
pd_df["pd_dec"] = parse_percent_or_decimal(pd_df[PD_COL], "pd_dec")

print(f"Missing PD: {pd_df['pd_dec'].isna().sum():,}")
print(f"Missing int_rate: {pd_df['int_rate_dec'].isna().sum():,}")
print(f"PD min/max/mean: {pd_df['pd_dec'].min():.4f}/{pd_df['pd_dec'].max():.4f}/{pd_df['pd_dec'].mean():.4f}")
print(f"Rate min/max/mean: {pd_df['int_rate_dec'].min():.4f}/{pd_df['int_rate_dec'].max():.4f}/{pd_df['int_rate_dec'].mean():.4f}")

active_pd = pd_df[pd_df["loan_status"].isin(ACTIVE_STATUSES)].copy()
exposure_col = next((c for c in EAD_COL_CANDIDATES if c in active_pd.columns), None)
if exposure_col is not None:
    active_pd["ead_weight"] = pd.to_numeric(active_pd[exposure_col], errors="coerce").clip(lower=0)
    active_pd.loc[active_pd["ead_weight"].isna() | (active_pd["ead_weight"] <= 0), "ead_weight"] = np.nan
    PD_AGG_METHOD = f"EAD-weighted ({exposure_col})"
else:
    active_pd["ead_weight"] = 1.0
    PD_AGG_METHOD = "Unweighted mean"
    print("NOTE: no exposure column found; using unweighted segment means.")

agg_base = active_pd.dropna(subset=["pd_dec", "int_rate_dec", "grade", "term_num", "purpose"]).copy()
segment_agg = (
    agg_base.groupby(["grade", "term_num", "purpose"]) 
    .apply(lambda x: pd.Series({
        "avg_pd": np.average(x["pd_dec"], weights=x["ead_weight"]),
        "std_pd": x["pd_dec"].std(),
        "n_pd": x["pd_dec"].count(),
        "avg_intrate": np.average(x["int_rate_dec"], weights=x["ead_weight"]),
        "std_intrate": x["int_rate_dec"].std(),
        "n_intrate": x["int_rate_dec"].count(),
    }))
    .reset_index()
)

df = df.merge(segment_agg, on=["grade", "term_num", "purpose"], how="left")
missing_pd = df["avg_pd"].isna()
missing_rate = df["avg_intrate"].isna()
print(f"L1 segments: {len(df):,}")
print(f"PD segments: {len(segment_agg):,}")
print(f"Missing avg_pd: {missing_pd.sum():,} ({missing_pd.mean():.1%})")
print(f"Missing avg_intrate: {missing_rate.sum():,} ({missing_rate.mean():.1%})")
print(f"EAD share missing avg_pd: {df.loc[missing_pd, 'ead'].sum()/df['ead'].sum():.2%}")
print(f"EAD share missing avg_intrate: {df.loc[missing_rate, 'ead'].sum()/df['ead'].sum():.2%}")

df = df.dropna(subset=["avg_pd", "avg_intrate"]).copy()


In [ ]:
# ── Fallback: if pd_predictions.csv has no int_rate, read from raw CSV ────────
# Uncomment this block if Step C raises a KeyError on 'int_rate'

# raw = pd.read_csv(
#     RAW_PATH,
#     usecols=["id", "loan_status", "grade", "term", "purpose", "int_rate"],
#     low_memory=False
# )
# raw["term_num"]     = raw["term"].astype(str).str.extract(r"(\d+)").astype(float)
# raw["int_rate_dec"] = raw["int_rate"] / 100
# active_raw = raw[raw["loan_status"].isin(ACTIVE_STATUSES)].copy()
#
# rate_agg = (
#     active_raw
#     .groupby(["grade", "term_num", "purpose"])["int_rate_dec"]
#     .agg(avg_intrate="mean", std_intrate="std", n_intrate="count")
#     .reset_index()
# )
# df = df.merge(rate_agg, on=["grade", "term_num", "purpose"], how="left")
# df = df.dropna(subset=["avg_intrate"]).copy()

print("Data ready. Sample:")
print(df[["grade","term_num","purpose","n","ead","avg_pd","el_rate",
          "avg_intrate","std_intrate","n_intrate"]].head(4).to_string(index=False))


## 3.1 Deliverable 1 — Required Rate Formula (Fully Decomposed)

r_req = funding cost + operating cost + PD × LGD + target margin.

PD comes directly from `pd_predictions.csv`; `el_12m_l1` is reference only.
Capital charge is not directly added to loan pricing; RAROC is used as a separate capital-efficiency check.


In [ ]:
df["el_premium"]   = df["avg_pd"] * LGD_CONST
df["el_rate"]      = df["el_premium"]
df["el_12m_model"] = df["el_premium"] * df["ead"]

df["r_req_raw"]        = FUNDING_COST + OP_COST + df["el_premium"] + TARGET_MARGIN
df["above_usury_cap"]  = df["r_req_raw"] > USURY_CAP
df["r_req"]            = df["r_req_raw"].clip(upper=USURY_CAP)
df["pricing_gap"]      = df["avg_intrate"] - df["r_req"]

df["ul"]         = df["ead"] * LGD_CONST * np.sqrt(df["avg_pd"] * (1 - df["avg_pd"]))
df["ec"]         = df["ul"] * K_VaR
df["revenue"]    = df["avg_intrate"] * df["ead"]
df["funding$"]   = FUNDING_COST * df["ead"]
df["opex$"]      = OP_COST * df["ead"]
df["el$"]        = df["el_12m_model"]
df["net_income"] = df["revenue"] - df["funding$"] - df["opex$"] - df["el$"]
df["raroc"]      = (df["net_income"] / df["ec"]).clip(-1, 2)


In [ ]:
# Visualise: r_req decomposition vs LC actual rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: segment scatter (bubble = EAD)
colors = [RED if g < 0 else GREEN for g in df["pricing_gap"]]
sizes  = (df["ead"] / df["ead"].max() * 300).clip(lower=5)
ax = axes[0]
ax.scatter(df["r_req"], df["avg_intrate"], c=colors, s=sizes, alpha=0.5, linewidths=0)
lims = [min(df["r_req"].min(), df["avg_intrate"].min()) - 0.01,
        max(df["r_req"].max(), df["avg_intrate"].max()) + 0.01]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("Required Rate (r_req)")
ax.set_ylabel("LC Actual Rate (r_lc)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_title("A. r_req vs r_lc (bubble = EAD)\nRed = Underpriced, Green = Overpriced", fontweight="bold")
ax.legend(handles=[
    Patch(color=RED,   label="Underpriced (r_lc < r_req)"),
    Patch(color=GREEN, label="Overpriced  (r_lc > r_req)"),
    plt.Line2D([0],[0], color="k", linestyle="--", label="Fair-price line"),
], fontsize=9)

# Panel B: stacked bar — r_req decomposition vs LC actual rate by grade
grade_decomp = (
    df.groupby("grade")
    .apply(lambda x: pd.Series({
        "funding": FUNDING_COST,
        "opex":    OP_COST,
        "el":      np.average(x["el_premium"], weights=x["ead"]),
        "margin":  TARGET_MARGIN,
        "r_lc":    np.average(x["avg_intrate"], weights=x["ead"]),
    }))
    .reindex(["A","B","C","D","E","F","G"])
)
ax2 = axes[1]
bottom = np.zeros(len(grade_decomp))
for col, color, label in [
    ("funding", "#4A90D9", f"Funding {FUNDING_COST:.1%}"),
    ("opex",    "#F5A623", f"OpEx {OP_COST:.1%}"),
    ("el",      "#E74C3C", "EL Premium (PD×LGD)"),
    ("margin",  "#27AE60", f"Margin {TARGET_MARGIN:.1%}"),
]:
    vals = grade_decomp[col].values * 100
    ax2.bar(grade_decomp.index, vals, bottom=bottom*100, color=color, label=label, alpha=0.85)
    bottom += grade_decomp[col].values
ax2.plot(grade_decomp.index, grade_decomp["r_lc"]*100,
         marker="o", color="black", linewidth=2, label="LC Actual Rate", zorder=5)
ax2.set_ylabel("Rate (%)")
ax2.set_title("B. r_req Decomposition vs LC Actual Rate\nby Grade", fontweight="bold")
ax2.legend(fontsize=9, loc="upper left")

plt.tight_layout()
plt.savefig(FIG_DIR / "pricing_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pricing_decomposition.png")


## 3.2 Deliverable 2 — Mispricing Heatmap (Grade × Term, 7×2)

Color = EAD-weighted mean pricing gap in **basis points** (bp).


In [ ]:
heatmap_bp = (
    df.groupby(["grade", "term_num"])
    .apply(lambda x: np.average(x["pricing_gap"], weights=x["ead"]))
    .unstack(level="term_num")
    .reindex(["A","B","C","D","E","F","G"]) * 10000
)
heatmap_bp.columns = [f"{int(c)}m" for c in heatmap_bp.columns]

ead_pivot = (
    df.groupby(["grade", "term_num"])["ead"].sum().unstack()
    .reindex(["A","B","C","D","E","F","G"]) / 1e6
)
ead_pivot.columns = [f"{int(c)}m" for c in ead_pivot.columns]

annot = pd.DataFrame(index=heatmap_bp.index, columns=heatmap_bp.columns, dtype=str)
for g in heatmap_bp.index:
    for t in heatmap_bp.columns:
        gap_v = heatmap_bp.loc[g, t]
        ead_v = ead_pivot.loc[g, t]
        if pd.notna(gap_v) and pd.notna(ead_v):
            sign = "+" if gap_v >= 0 else ""
            annot.loc[g, t] = f"{sign}{gap_v:.0f}bp\n${ead_v:.0f}M"
        else:
            annot.loc[g, t] = "N/A"

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    heatmap_bp, annot=annot, fmt="", cmap="RdYlGn", center=0,
    ax=ax, linewidths=1, linecolor="white",
    cbar_kws={"label": "Pricing Gap (bp) = r_lc − r_req"}
)
ax.set_title("Mispricing Heatmap: Grade × Term (bp)", fontweight="bold")
ax.set_xlabel("Loan Term")
ax.set_ylabel("LC Grade")
plt.tight_layout()
plt.savefig(FIG_DIR / "mispricing_heatmap_grade_term.png", dpi=150, bbox_inches="tight")
plt.show()


## 3.3 Deliverable 3 — Three-Bucket Strategy with Significance Testing

Priority rules:
1) `above_usury_cap` => Decline / Do not originate.
2) `pricing_gap < REPRICE_THRESH` and `raroc < HURDLE_RATE` => Reprice.
3) `pricing_gap > GROW_THRESH` and `raroc > HURDLE_RATE` => Grow.
4) Else => Hold / Monitor.


In [ ]:
df["se_intrate"] = df["std_intrate"] / np.sqrt(df["n_intrate"])
df["intrate_ci_lower"] = df["avg_intrate"] - Z_SCORE * df["se_intrate"]
df["intrate_ci_upper"] = df["avg_intrate"] + Z_SCORE * df["se_intrate"]
df["gap_ci_lower"] = df["intrate_ci_lower"] - df["r_req"]
df["gap_ci_upper"] = df["intrate_ci_upper"] - df["r_req"]

def assign_bucket(row):
    if row["above_usury_cap"]:
        return "Decline / Do not originate"
    if (row["pricing_gap"] < REPRICE_THRESH) and (row["raroc"] < HURDLE_RATE):
        return "Reprice"
    if (row["pricing_gap"] > GROW_THRESH) and (row["raroc"] > HURDLE_RATE):
        return "Grow"
    return "Hold / Monitor"

df["bucket"] = df.apply(assign_bucket, axis=1)


In [ ]:
# Action table (grade × term level)
action_table = (
    df.groupby(["grade", "term_num", "bucket"])
    .apply(lambda x: pd.Series({
        "n_loans":           x["n"].sum(),
        "ead_$M":            x["ead"].sum() / 1e6,
        "r_lc":              np.average(x["avg_intrate"],  weights=x["ead"]),
        "r_req":             np.average(x["r_req"],        weights=x["ead"]),
        "gap_bp":            np.average(x["pricing_gap"],  weights=x["ead"]) * 10000,
        "gap_ci_lower_bp":   np.average(x["gap_ci_lower"], weights=x["ead"]) * 10000,
        "gap_ci_upper_bp":   np.average(x["gap_ci_upper"], weights=x["ead"]) * 10000,
    }))
    .reset_index()
)

# Incremental NII for Reprice bucket
action_table["incremental_NII_$M"] = np.where(
    action_table["bucket"] == "Reprice",
    (action_table["r_req"] - action_table["r_lc"]) * action_table["ead_$M"],
    0.0
)

# Formatted display
at_show = action_table.copy()
at_show["r_lc"]  = at_show["r_lc"].map("{:.2%}".format)
at_show["r_req"] = at_show["r_req"].map("{:.2%}".format)
at_show["gap_bp"] = at_show["gap_bp"].map("{:.0f}bp".format)
at_show["CI_95%"] = at_show.apply(
    lambda r: f"[{r['gap_ci_lower_bp']:.0f}, {r['gap_ci_upper_bp']:.0f}]bp", axis=1
)
at_show["ead_$M"] = at_show["ead_$M"].map("${:.1f}M".format)
at_show["incremental_NII_$M"] = at_show["incremental_NII_$M"].map("${:.1f}M".format)

print("=== Segment-Level Action Table (grade × term) ===")
display_cols = ["grade","term_num","bucket","n_loans","ead_$M",
                "r_lc","r_req","gap_bp","CI_95%","incremental_NII_$M"]
print(at_show[display_cols].sort_values(["bucket","gap_bp"]).to_string(index=False))
action_table.to_csv(OUT_DIR / "l2_action_table.csv", index=False)
print("\nSaved: l2_action_table.csv")


In [ ]:
# Visualise three-bucket strategy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bucket_colors = {"Grow": GREEN, "Hold / Monitor": TEAL, "Reprice": ORANGE, "Decline / Do not originate": RED}

# Panel A: EAD by bucket
ead_by_bucket = (
    df.groupby("bucket")["ead"].sum()
    .reindex(["Grow","Hold / Monitor","Reprice","Decline / Do not originate"]) / 1e9
)
bars = axes[0].bar(
    ead_by_bucket.index, ead_by_bucket.values,
    color=[bucket_colors[b] for b in ead_by_bucket.index],
    edgecolor="white", alpha=0.9
)
for bar, v in zip(bars, ead_by_bucket.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.03,
                 f"${v:.2f}B", ha="center", fontsize=10, fontweight="bold")
axes[0].set_ylabel("EAD ($B)")
axes[0].set_title("A. EAD by Strategy Bucket\n(2018Q4 Active Portfolio)", fontweight="bold")

# Panel B: gap + CI by segment (sorted by gap)
df_sorted = df.sort_values("pricing_gap").reset_index(drop=True)
colors_b = [bucket_colors[b] for b in df_sorted["bucket"]]
axes[1].scatter(
    df_sorted.index, df_sorted["pricing_gap"]*100,
    c=colors_b, s=(df_sorted["ead"]/df_sorted["ead"].max()*150).clip(lower=5),
    alpha=0.7, zorder=3
)
# CI bars
for i, row in df_sorted.iterrows():
    if row["ci_reliable"]:
        axes[1].plot(
            [i, i],
            [row["gap_ci_lower"]*100, row["gap_ci_upper"]*100],
            color=bucket_colors[row["bucket"]], alpha=0.3, linewidth=1
        )
axes[1].axhline(REPRICE_THRESH*100, color=ORANGE, linestyle="--",
                label=f"Reprice threshold ({REPRICE_THRESH*100:.0f}bp)")
axes[1].axhline(GROW_THRESH*100, color=GREEN, linestyle="--",
                label=f"Grow threshold (+{GROW_THRESH*100:.0f}bp)")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Segments (sorted by gap)")
axes[1].set_ylabel("Pricing Gap (bp)")
axes[1].set_title("B. Pricing Gap with 95% CI\n(thin bar = confidence interval)",
                  fontweight="bold")
axes[1].legend(fontsize=9, handles=[
    Patch(color=GREEN,  label="Grow"),
    Patch(color=TEAL,   label="Hold / Monitor"),
    Patch(color=ORANGE, label="Reprice"),
    Patch(color=RED,    label="Decline / Do not originate"),
    plt.Line2D([0],[0], color=ORANGE, linestyle="--", label=f"−50bp threshold"),
    plt.Line2D([0],[0], color=GREEN,  linestyle="--", label=f"+50bp threshold"),
])

plt.tight_layout()
plt.savefig(FIG_DIR / "three_bucket_strategy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: three_bucket_strategy.png")


## 3.4 Deliverable 4 — RAROC by Segment (The Killer Slide)

$$\text{RAROC} = \frac{\text{Revenue} - \text{Funding} - \text{OpEx} - \text{EL}}{\text{Economic Capital}}$$

**Economic Capital (Basel IRB simplified):**
$$UL = EAD \times LGD \times \sqrt{PD(1-PD)} \qquad EC = UL \times k \quad (k=3.5)$$

To be replaced by L4's output once available. The relative ranking across segments
is robust to the choice of k.

**Story:** LC uses good borrowers to subsidise bad ones.
Grade A/B RAROC >> 12% (creating value); Grade F/G RAROC << 12% (destroying value).
This is the quantitative proof behind the three-bucket strategy and the capstone RAROC waterfall.

In [ ]:
raroc_gt = (
    df.groupby(["grade", "term_num"])
    .apply(lambda x: pd.Series({
        "ead_$B":        x["ead"].sum() / 1e9,
        "avg_pd":        np.average(x["avg_pd"],      weights=x["ead"]),
        "r_lc":          np.average(x["avg_intrate"], weights=x["ead"]),
        "el_rate":       x["el_12m_model"].sum() / x["ead"].sum(),
        "ec_$B":         x["ec"].sum() / 1e9,
        "net_income_$M": x["net_income"].sum() / 1e6,
        "raroc":         x["net_income"].sum() / x["ec"].sum(),
    }))
    .reset_index()
)


In [ ]:
# RAROC horizontal bar chart — the killer slide
fig, ax = plt.subplots(figsize=(10, 8))

def raroc_color(r):
    if r >= HURDLE_RATE:        return GREEN
    elif r >= HURDLE_RATE / 2:  return ORANGE
    else:                       return RED

colors = [raroc_color(r) for r in raroc_gt["raroc"]]
bars = ax.barh(raroc_gt["segment"], raroc_gt["raroc"]*100,
               color=colors, edgecolor="white", alpha=0.9)
for bar, v in zip(bars, raroc_gt["raroc"]):
    xpos = v*100 + 0.3 if v >= 0 else v*100 - 0.3
    ha = "left" if v >= 0 else "right"
    ax.text(xpos, bar.get_y()+bar.get_height()/2,
            f"{v:.1%}", va="center", ha=ha, fontsize=9, fontweight="bold")

ax.axvline(HURDLE_RATE*100, color="black", linewidth=2, linestyle="--",
           label=f"Hurdle rate = {HURDLE_RATE:.0%}")
ax.axvline(HURDLE_RATE*100/2, color="gray", linewidth=1, linestyle=":")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("RAROC (%)")
ax.set_title(
    "RAROC by Grade × Term — 2018Q4 Active Portfolio\n"
    "Green ≥ 12% (creating value) | Orange 6–12% (marginal) | Red < 6% (destroying value)",
    fontweight="bold"
)
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.tight_layout()
plt.savefig(FIG_DIR / "raroc_by_segment.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: raroc_by_segment.png")


## 3.5 Deliverable 5 — Repricing Impact Simulation (Pre / Post / Delta)

This is a counterfactual policy test for future origination / portfolio selection, not literal repricing of fixed-rate loans.


In [ ]:
sim = df.copy()
sim["rate_post"] = sim["avg_intrate"].copy()
sim["ead_post"]  = sim["ead"].copy()

sim.loc[sim["bucket"]=="Reprice", "rate_post"] = sim.loc[sim["bucket"]=="Reprice", "r_req"]
sim.loc[sim["bucket"]=="Decline / Do not originate", "ead_post"]  = 0.0

sim["revenue_post"]    = sim["rate_post"]  * sim["ead_post"]
sim["funding_post"]    = FUNDING_COST      * sim["ead_post"]
sim["opex_post"]       = OP_COST           * sim["ead_post"]
sim["el_post"]         = sim["el_premium"] * sim["ead_post"]
sim["net_income_post"] = sim["revenue_post"] - sim["funding_post"] - sim["opex_post"] - sim["el_post"]
sim["ec_post"]         = (sim["ul"] / sim["ead"].clip(lower=1)) * sim["ead_post"] * K_VaR


In [ ]:
# Visualise Pre vs Post
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Portfolio RAROC
r_pre, r_post = pre["Portfolio RAROC"], post["Portfolio RAROC"]
bars = axes[0].bar(["Current", "Post-Strategy"], [r_pre*100, r_post*100],
                   color=[ORANGE, GREEN], edgecolor="white", width=0.4)
axes[0].axhline(HURDLE_RATE*100, color="black", linewidth=2,
                linestyle="--", label=f"Hurdle {HURDLE_RATE:.0%}")
for bar, v in zip(bars, [r_pre, r_post]):
    axes[0].text(bar.get_x()+bar.get_width()/2, v*100+0.2,
                 f"{v:.1%}", ha="center", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Portfolio RAROC")
axes[0].set_title("A. Portfolio RAROC\nCurrent vs Post-Strategy", fontweight="bold")
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Panel B: Net Income
ni_pre, ni_post = pre["Net Income ($M)"], post["Net Income ($M)"]
bars = axes[1].bar(["Current", "Post-Strategy"], [ni_pre, ni_post],
                   color=[ORANGE, GREEN], edgecolor="white", width=0.4)
for bar, v in zip(bars, [ni_pre, ni_post]):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+1,
                 f"${v:.1f}M", ha="center", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Net Income ($M)")
axes[1].set_title("B. Annual Net Income\nCurrent vs Post-Strategy", fontweight="bold")

# Panel C: Expected Loss
el_pre, el_post = pre["Expected Loss ($M)"], post["Expected Loss ($M)"]
bars = axes[2].bar(["Current", "Post-Strategy"], [el_pre, el_post],
                   color=[RED, TEAL], edgecolor="white", width=0.4)
for bar, v in zip(bars, [el_pre, el_post]):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+0.5,
                 f"${v:.1f}M", ha="center", fontsize=11, fontweight="bold")
axes[2].set_ylabel("Expected Loss ($M)")
axes[2].set_title("C. Annual Expected Loss\nCurrent vs Post-Strategy", fontweight="bold")

plt.tight_layout()
plt.savefig(FIG_DIR / "repricing_impact.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: repricing_impact.png")


## 3.6 Validation: Pricing Framework Internal Consistency

Five checks. No future default data needed.

**Check 2 note:** Validation benchmark is **mean** realized loss across 4 quarters of 2016 data,
not total. Total realized loss = EL + UL; pricing only targets EL. Pooling multiple quarters
lets UL noise cancel, so the multi-period mean approximates the true EL.

In [ ]:
print("Validation checks:")
print(f"PD column detected: {PD_COL}")
print(f"PD aggregation method: {PD_AGG_METHOD}")
print(f"Above usury cap segments: {df['above_usury_cap'].sum()} / {len(df)}")


## 3.7 Save L2 Outputs (for L3 / L4)

In [ ]:
l2_output = {
    "pd_column_detected": PD_COL,
    "pd_aggregation_method": PD_AGG_METHOD,
    "usury_cap": USURY_CAP,
    "active_ead_$B": float(df["ead"].sum() / 1e9),
    "portfolio_avg_r_lc": float((df["avg_intrate"]*df["ead"]).sum() / df["ead"].sum()),
    "portfolio_avg_r_req": float((df["r_req"]*df["ead"]).sum() / df["ead"].sum()),
    "portfolio_avg_gap": float((df["pricing_gap"]*df["ead"]).sum() / df["ead"].sum()),
    "portfolio_raroc_current": float(df["net_income"].sum() / df["ec"].sum()),
    "portfolio_raroc_post": float(sim["net_income_post"].sum() / sim["ec_post"].sum()),
    "el_current_$M": float(df["el_12m_model"].sum() / 1e6),
    "el_post_$M": float(sim["el_post"].sum() / 1e6)
}

with open(OUT_DIR / "l2_pricing.json", "w") as f:
    json.dump(l2_output, f, indent=2)

df[[
    "grade","term_num","purpose","n","ead","avg_pd","avg_intrate",
    "el_premium","el_12m_model","el_12m_l1","r_req_raw","above_usury_cap",
    "r_req","pricing_gap","bucket","ec","raroc"
]].to_csv(OUT_DIR / "l2_segment_table.csv", index=False)

print(f"Saved: {OUT_DIR / 'l2_pricing.json'}")
print(f"Saved: {OUT_DIR / 'l2_segment_table.csv'}")
